In [ ]:
import os, sys, subprocess
from pathlib import Path

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. In Colab: Runtime → Change runtime type → Hardware accelerator: T4 GPU")

gpu_name = torch.cuda.get_device_name(0)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ANRA = Path("/content/drive/MyDrive/AnRa")
DRIVE_ANRA.mkdir(parents=True, exist_ok=True)

REPO = None
for candidate in sorted(Path("/content").iterdir()):
    if candidate.is_dir() and (candidate / "anra_paths.py").exists():
        REPO = candidate
        break

GITHUB_URL = "https://github.com/YOUR_USERNAME/An-Ra-the-new-AGI.git"
if REPO is None:
    if "YOUR_USERNAME" in GITHUB_URL:
        raise RuntimeError(
            "Edit GITHUB_URL in Cell 1 first. Replace YOUR_USERNAME with your actual GitHub username/org, "
            "then re-run Cell 1."
        )
    print("Cloning repository...")
    result = subprocess.run(["git", "clone", "--quiet", GITHUB_URL], cwd="/content", capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{result.stderr}")
    for candidate in sorted(Path("/content").iterdir()):
        if candidate.is_dir() and (candidate / "anra_paths.py").exists():
            REPO = candidate
            break
    if REPO is None:
        raise RuntimeError("Clone finished but repository folder was not found under /content.")
else:
    pull = subprocess.run(["git", "pull", "--quiet"], cwd=str(REPO), capture_output=True, text=True)
    if pull.returncode != 0:
        print(f"Git pull warning: {pull.stderr.strip()}")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)

try:
    from anra_paths import inject_all_paths
    inject_all_paths()
except Exception as exc:
    raise RuntimeError(f"Failed importing anra_paths after setup: {exc}")

Path("/tmp/anra_repo_path.txt").write_text(str(REPO), encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tokenizers", "sympy", "networkx", "accelerate"],
    check=True,
)

print("=" * 60)
print(f"GPU   : {gpu_name}")
print(f"REPO  : {REPO}")
print(f"Drive : {DRIVE_ANRA}")
print("=" * 60)


In [ ]:
import os, sys
from pathlib import Path

def _auto_find_repo():
    f = Path("/tmp/anra_repo_path.txt")
    if f.exists():
        p = Path(f.read_text().strip())
        if (p / "anra_paths.py").exists():
            return p
    for c in sorted(Path("/content").iterdir()):
        if c.is_dir() and (c / "anra_paths.py").exists():
            return c
    raise RuntimeError("Run Cell 1 first.")

REPO = _auto_find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
from anra_paths import inject_all_paths
inject_all_paths()

import shutil
import torch
from anra_paths import get_v2_checkpoint, V2_TOKENIZER_FILE

DRIVE_ANRA = Path("/content/drive/MyDrive/AnRa")
DRIVE_BRAIN = DRIVE_ANRA / "anra_v2_brain.pt"
DRIVE_TOKENIZER = DRIVE_ANRA / "tokenizer_v2.json"

LOCAL_BRAIN = get_v2_checkpoint("brain")
LOCAL_TOKENIZER = V2_TOKENIZER_FILE
LOCAL_BRAIN.parent.mkdir(parents=True, exist_ok=True)
LOCAL_TOKENIZER.parent.mkdir(parents=True, exist_ok=True)

if DRIVE_BRAIN.exists():
    shutil.copy2(DRIVE_BRAIN, LOCAL_BRAIN)
    print(f"✓ Restored brain checkpoint to {LOCAL_BRAIN}")
else:
    print("○ Drive brain checkpoint not found yet")

if DRIVE_TOKENIZER.exists():
    shutil.copy2(DRIVE_TOKENIZER, LOCAL_TOKENIZER)
    print(f"✓ Restored tokenizer_v2.json to {LOCAL_TOKENIZER}")
else:
    print("○ Drive tokenizer_v2.json not found yet")

if LOCAL_BRAIN.exists():
    try:
        ckpt = torch.load(LOCAL_BRAIN, map_location="cpu", weights_only=False)
        step = int(ckpt.get("step", 0))
        loss = float(ckpt.get("best_loss", float("inf")))
        sessions = int(ckpt.get("sessions_completed", 0))
        print(f"Checkpoint metadata: step={step} loss={loss:.4f} sessions={sessions}")
    except Exception as exc:
        print(f"Checkpoint exists but metadata read failed: {exc}")

print("Run Cell 3 next")


In [ ]:
import os, sys
from pathlib import Path

def _auto_find_repo():
    f = Path("/tmp/anra_repo_path.txt")
    if f.exists():
        p = Path(f.read_text().strip())
        if (p / "anra_paths.py").exists():
            return p
    for c in sorted(Path("/content").iterdir()):
        if c.is_dir() and (c / "anra_paths.py").exists():
            return c
    raise RuntimeError("Run Cell 1 first.")

REPO = _auto_find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
from anra_paths import inject_all_paths
inject_all_paths()

import importlib
import torch
from anra_paths import PROJECT_ROOT, get_v2_checkpoints_dir, get_v2_checkpoint, V2_TOKENIZER_FILE

failures = []

def _ok(label, detail=""):
    suffix = f" — {detail}" if detail else ""
    print(f"✓ {label}{suffix}")

def _fail(label, detail):
    failures.append((label, detail))
    print(f"✗ {label} — {detail}")

if torch.cuda.is_available():
    _ok("GPU availability", torch.cuda.get_device_name(0))
else:
    _fail("GPU availability", "Enable GPU in Colab Runtime settings")

_ok("PyTorch version", torch.__version__)
_ok("anra_paths import", f"PROJECT_ROOT={PROJECT_ROOT}, v2_dir={get_v2_checkpoints_dir()}")

try:
    importlib.import_module("training.train_unified")
    _ok("training.train_unified importable")
except Exception as exc:
    _fail("training.train_unified importable", f"Re-run Cell 1 and verify dependencies. Error: {exc}")

if V2_TOKENIZER_FILE.exists():
    _ok("tokenizer_v2.json exists", str(V2_TOKENIZER_FILE))
else:
    _fail("tokenizer_v2.json exists", "Run Cell 2 to restore or train once to generate tokenizer")

ckpt_path = get_v2_checkpoint("brain")
if ckpt_path.exists():
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        step = int(ckpt.get("step", 0))
        loss = float(ckpt.get("best_loss", float("inf")))
        _ok("anra_v2_brain.pt exists", f"step={step} loss={loss:.4f}")
    except Exception as exc:
        _fail("anra_v2_brain.pt metadata", f"Checkpoint unreadable: {exc}")
else:
    _fail("anra_v2_brain.pt exists", "Run Cell 2 restore or train in Cell 4")

print()
if failures:
    print("Health check failures:")
    for label, detail in failures:
        print(f"  ✗ {label}: {detail}")
    print("Fix the above items, then re-run Cell 3.")
else:
    print("✓ All checks passed. Run Cell 4.")


In [ ]:
import os, sys
from pathlib import Path

def _auto_find_repo():
    f = Path("/tmp/anra_repo_path.txt")
    if f.exists():
        p = Path(f.read_text().strip())
        if (p / "anra_paths.py").exists():
            return p
    for c in sorted(Path("/content").iterdir()):
        if c.is_dir() and (c / "anra_paths.py").exists():
            return c
    raise RuntimeError("Run Cell 1 first.")

REPO = _auto_find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
from anra_paths import inject_all_paths
inject_all_paths()

import subprocess

SESSION_MINUTES = 30

print(f"Starting training session for {SESSION_MINUTES} minutes...")
print("-" * 60)

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "training.train_unified",
        "--mode",
        "session",
        "--session_minutes",
        str(SESSION_MINUTES),
    ],
    cwd=str(REPO),
    env={**os.environ, "PYTHONPATH": str(REPO), "PYTHONUNBUFFERED": "1"},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    process.terminate()
    print("\nTraining interrupted by user.")

process.wait()
print("-" * 60)
if process.returncode == 0:
    print("Run Cell 5 to save")
else:
    print(f"Training failed with return code {process.returncode}")
    print("Most common fix: re-run Cell 1")


In [ ]:
import os, sys
from pathlib import Path

def _auto_find_repo():
    f = Path("/tmp/anra_repo_path.txt")
    if f.exists():
        p = Path(f.read_text().strip())
        if (p / "anra_paths.py").exists():
            return p
    for c in sorted(Path("/content").iterdir()):
        if c.is_dir() and (c / "anra_paths.py").exists():
            return c
    raise RuntimeError("Run Cell 1 first.")

REPO = _auto_find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
from anra_paths import inject_all_paths
inject_all_paths()

import shutil
import torch
from anra_paths import get_v2_checkpoint, V2_TOKENIZER_FILE

DRIVE_ANRA = Path("/content/drive/MyDrive/AnRa")
DRIVE_ANRA.mkdir(parents=True, exist_ok=True)

files = [
    (get_v2_checkpoint("brain"), DRIVE_ANRA / "anra_v2_brain.pt"),
    (V2_TOKENIZER_FILE, DRIVE_ANRA / "tokenizer_v2.json"),
    (REPO / "output" / "v2" / "anra_v2_eval_summary.json", DRIVE_ANRA / "anra_v2_eval_summary.json"),
]

for src, dst in files:
    if src.exists():
        shutil.copy2(src, dst)
        print(f"✓ Saved {src.name} ({src.stat().st_size // 1024} KB)")
    else:
        if src.name == "anra_v2_eval_summary.json":
            print("○ Eval summary missing locally (skipped)")
        else:
            print(f"✗ Missing required local file: {src}")

ckpt = DRIVE_ANRA / "anra_v2_brain.pt"
if ckpt.exists():
    try:
        data = torch.load(ckpt, map_location="cpu", weights_only=False)
        step = int(data.get("step", 0))
        loss = float(data.get("best_loss", float("inf")))
        print(f"Drive checkpoint confirmation: step={step} loss={loss:.4f}")
    except Exception as exc:
        print(f"Drive checkpoint saved but metadata read failed: {exc}")

print("✓ Safe to close Colab.")


In [ ]:
import os, sys
from pathlib import Path

def _auto_find_repo():
    f = Path("/tmp/anra_repo_path.txt")
    if f.exists():
        p = Path(f.read_text().strip())
        if (p / "anra_paths.py").exists():
            return p
    for c in sorted(Path("/content").iterdir()):
        if c.is_dir() and (c / "anra_paths.py").exists():
            return c
    raise RuntimeError("Run Cell 1 first.")

REPO = _auto_find_repo()
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
from anra_paths import inject_all_paths
inject_all_paths()

import subprocess

print("Starting milestone training session (~90 min)...")
print("-" * 60)

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "training.train_unified",
        "--mode",
        "train",
        "--session_minutes",
        "90",
    ],
    cwd=str(REPO),
    env={**os.environ, "PYTHONPATH": str(REPO), "PYTHONUNBUFFERED": "1"},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:
    process.terminate()
    print("\nMilestone session interrupted by user.")

process.wait()
print("-" * 60)
if process.returncode == 0:
    print("Milestone session complete. Run Cell 5 to save.")
else:
    print(f"Milestone training failed with return code {process.returncode}")
    print("Most common fix: re-run Cell 1")
